# Actividad 9 · Entry points del pipeline de scikit-learn

Curso **Machine Learning Engineering (MLE/MLOps)**, Universidad del Valle de Guatemala.
Diego Linares · Andy Fuentes · Christian Echeverria · Diederich Solis

Repositorio: https://github.com/DiegoLinares11/Actividad-9-MLOPS

Desde la terminal, el pipeline se corre con comandos (`act9-datos`, `act9-entrenar`, ...). Este
notebook hace lo mismo desde Python para ver qué hay detrás de esos comandos: dónde quedan
registrados, cómo se cargan y cómo se descubren los modelos que vienen como plugins.

Requisito: haber instalado el paquete en el ambiente del kernel (`pip install -e ".[notebook]"`).

## 1. El archivo que escribe el instalador

Al instalar el paquete, pip (o uv, Poetry, conda) lee `[project.scripts]` y
`[project.entry-points]` del `pyproject.toml` y los copia a `entry_points.txt`, dentro de la
carpeta `.dist-info` del paquete instalado. Cada sección es un **grupo** y cada línea un
**nombre** con su **referencia** `modulo:objeto`.

In [1]:
from importlib.metadata import distribution, entry_points

dist = distribution("act9-pipeline")
print(dist.read_text("entry_points.txt"))

[act9_pipeline.modelos]
hist_gradient_boosting = act9_pipeline.modelos:hist_gradient_boosting
random_forest = act9_pipeline.modelos:random_forest
regresion_logistica = act9_pipeline.modelos:regresion_logistica
svc = act9_pipeline.modelos:svc

[console_scripts]
act9 = act9_pipeline.cli:main
act9-datos = act9_pipeline.cli.datos:main
act9-demo = act9_pipeline.cli.demo:main
act9-entrenar = act9_pipeline.cli.entrenar:main
act9-evaluar = act9_pipeline.cli.evaluar:main
act9-predecir = act9_pipeline.cli.predecir:main



In [2]:
import pandas as pd

pd.DataFrame(
    [{"grupo": ep.group, "nombre": ep.name, "módulo": ep.module, "objeto": ep.attr}
     for ep in dist.entry_points]
)

,grupo,nombre,módulo,objeto
0,act9_pipeline.modelos,hist_gradient_boosting,act9_pipeline.modelos,hist_gradient_boosting
1,act9_pipeline.modelos,random_forest,act9_pipeline.modelos,random_forest
2,act9_pipeline.modelos,regresion_logistica,act9_pipeline.modelos,regresion_logistica
3,act9_pipeline.modelos,svc,act9_pipeline.modelos,svc
4,console_scripts,act9,act9_pipeline.cli,main
5,console_scripts,act9-datos,act9_pipeline.cli.datos,main
6,console_scripts,act9-demo,act9_pipeline.cli.demo,main
7,console_scripts,act9-entrenar,act9_pipeline.cli.entrenar,main
8,console_scripts,act9-evaluar,act9_pipeline.cli.evaluar,main
9,console_scripts,act9-predecir,act9_pipeline.cli.predecir,main


## 2. Un comando es solo una función

`console_scripts` es el grupo especial: por cada línea el instalador crea un ejecutable (en
Windows, `Scripts\act9-datos.exe`). Lo único que hace ese ejecutable es importar la función y
llamarla sin argumentos. Aquí se hace lo mismo a mano con `EntryPoint.load()`; lo que devuelve la
función es el código de salida del proceso.

In [3]:
import tempfile
from pathlib import Path

ep = entry_points(group="console_scripts")["act9-datos"]
act9_datos = ep.load()
print(ep.value, "->", act9_datos)

carpeta = Path(tempfile.mkdtemp(prefix="act9-notebook-"))
codigo = act9_datos(["--salida", str(carpeta)])
print("código de salida:", codigo)

act9_pipeline.cli.datos:main -> <function main at 0x000001F98424DC70>
[act9-datos] leyendo CSV empaquetado en act9_pipeline/datasets
[act9-datos] crudo: 151 filas -> limpio: 144 partidos
[act9-datos] clases: Home Win=71  Away Win=48  Draw=25
[act9-datos] train=115  test=29  (estratificado, seed=42)
[act9-datos] escrito C:\Users\dlinares\AppData\Local\Temp\act9-notebook-teyi8mw7\train.csv y C:\Users\dlinares\AppData\Local\Temp\act9-notebook-teyi8mw7\test.csv
código de salida: 0


## 3. Las familias de modelos son plugins

El grupo `act9_pipeline.modelos` no crea ejecutables. Solo deja un registro en la metadata
que `act9-entrenar` lee al arrancar. Cualquier paquete instalado que se registre en ese grupo
agrega una familia a la búsqueda de hiperparámetros sin cambiar este código.

In [4]:
from act9_pipeline import descubrir_modelos

familias = descubrir_modelos()
pd.DataFrame(
    [{"familia": f.nombre, "referencia": f.referencia, "paquete": f.distribucion,
      "hiperparámetros": ", ".join(f.espacio)}
     for f in familias.values()]
)

,familia,referencia,paquete,hiperparámetros
0,hist_gradient_boosting,act9_pipeline.modelos:hist_gradient_boosting,act9-pipeline 0.1.0,"learning_rate, max_iter, max_leaf_nodes, min_s..."
1,random_forest,act9_pipeline.modelos:random_forest,act9-pipeline 0.1.0,"n_estimators, max_depth, min_samples_leaf, max..."
2,regresion_logistica,act9_pipeline.modelos:regresion_logistica,act9-pipeline 0.1.0,"C, class_weight"
3,svc,act9_pipeline.modelos:svc,act9-pipeline 0.1.0,"C, gamma, kernel, class_weight"


## 4. Entrenar y evaluar con las mismas funciones

`act9-entrenar` arma un `RandomizedSearchCV` con un espacio por familia descubierta (las
mismas de la Actividad 3) y guarda el mejor pipeline. `act9-evaluar` lo mide en el conjunto de
prueba.

In [5]:
from act9_pipeline.cli import entrenar, evaluar

codigo = entrenar.main(["--entrada", str(carpeta / "train.csv"), "--salida", str(carpeta),
                        "--diagrama", ""])
print("código de salida:", codigo)

[act9-entrenar] familias descubiertas en 'act9_pipeline.modelos': hist_gradient_boosting, random_forest, regresion_logistica, svc
[act9-entrenar] familias en la busqueda: hist_gradient_boosting, random_forest, regresion_logistica, svc
[act9-entrenar] 115 partidos de entrenamiento leidos de C:\Users\dlinares\AppData\Local\Temp\act9-notebook-teyi8mw7\train.csv
[act9-entrenar] RandomizedSearchCV: 60 combinaciones x 5 folds, metrica f1_macro


[act9-entrenar] listo en 21.5 s | mejor f1_macro en CV = 0.652 (+/- 0.076)
[act9-entrenar] ganador: svc -> SVC (plugin de act9-pipeline 0.1.0)
           modelo__C = 38.765111709116326
           modelo__class_weight = balanced
           modelo__gamma = 0.004149851045848953
           modelo__kernel = rbf
           preprocesamiento__numericas__imputar__strategy = mean
           seleccion__k = all
[act9-entrenar] modelo guardado en C:\Users\dlinares\AppData\Local\Temp\act9-notebook-teyi8mw7\modelo.joblib (67 KB)
código de salida: 0


In [6]:
codigo = evaluar.main(["--modelo", str(carpeta / "modelo.joblib"),
                       "--entrada", str(carpeta / "test.csv"), "--salida", str(carpeta),
                       "--umbral", "0.5"])
print("código de salida:", codigo)

[act9-evaluar] modelo SVC (svc), entrenado con act9-entrenar el 2026-10-05T01:12:44+00:00
[act9-evaluar] 29 partidos de prueba leidos de C:\Users\dlinares\AppData\Local\Temp\act9-notebook-teyi8mw7\test.csv

RESULTADO - accuracy=0.655  f1_macro=0.604
Baseline (clase mayoritaria): accuracy=0.483  f1_macro=0.217

              precision    recall  f1-score   support

    Away Win      0.692     0.900     0.783        10
        Draw      0.286     0.400     0.333         5
    Home Win      0.889     0.571     0.696        14

    accuracy                          0.655        29
   macro avg      0.622     0.624     0.604        29
weighted avg      0.717     0.655     0.663        29

Matriz de confusion:
                pred: Away Win  pred: Draw  pred: Home Win
real: Away Win               9           1               0
real: Draw                   2           2               1
real: Home Win               2           4               8

[act9-evaluar] metricas guardadas en C:\Users\dli

## 5. Diagrama del pipeline ganador

In [7]:
from sklearn import set_config

from act9_pipeline import cargar_modelo

set_config(display="diagram")
modelo, metadatos = cargar_modelo(carpeta / "modelo.joblib")
print("Familia ganadora:", metadatos["familia_ganadora"], "| plugin de", metadatos["plugin_de"])
modelo

Familia ganadora: svc | plugin de act9-pipeline 0.1.0


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocesamiento', ...), ('seleccion', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](3,)","['Away Win','Draw','Home Win']"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](12,)","['home_shots_on_target_pct','away_shots_on_target_pct','home_saves_pct', ...,'away_saves','home_team','away_team']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,12
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numericas', ...), ('porcentajes', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-

## 6. Agregar una familia sin tocar el paquete

El plugin de ejemplo `plugins/act9-modelo-knn` es otro paquete, con su propio `pyproject.toml`,
que solo se registra en el grupo `act9_pipeline.modelos`:

```toml
[project.entry-points."act9_pipeline.modelos"]
knn = "act9_modelo_knn:knn"
```

Después de `pip install -e plugins/act9-modelo-knn` (o `make plugin-knn`), la celda de la
sección 3 muestra cinco familias y `act9-entrenar --familias knn` entrena con KNN.